In [ ]:
import marimo as mo

In [ ]:
from gdrive_fsspec import GoogleDriveFileSystem

fs = GoogleDriveFileSystem(use_listings_cache=False, skip_instance_cache=True, auth_kwargs={"use_local_webserver": False})
mo.output.clear_console()

In [ ]:
import json
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

# Cargar credenciales OAuth desde el archivo de configuración
with open("/home/marimo/.config/pydata/pydata_google_credentials.json") as f:
    creds_data = json.load(f)

# Construir y refrescar las credenciales
creds = Credentials(
    token=creds_data.get("token"),
    refresh_token=creds_data.get("refresh_token"),
    token_uri=creds_data.get("token_uri"),
    client_id=creds_data.get("client_id"),
    client_secret=creds_data.get("client_secret"),
    scopes=creds_data.get("scopes", ["https://www.googleapis.com/auth/drive"]),
)
creds.refresh(Request())

# Crear el servicio de Google Drive
drive_service = build("drive", "v3", credentials=creds)
print("✅ Servicio de Google Drive creado correctamente")

In [ ]:
# Buscar la carpeta "silicon_sampling" en Google Drive
silicon_sampling_folder_id = None

_results = drive_service.files().list(
    q="name='silicon_sampling' and mimeType='application/vnd.google-apps.folder'",
    fields="files(id, name)",
    pageSize=10,
).execute()

_folders = _results.get("files", [])
if _folders:
    silicon_sampling_folder_id = _folders[0]["id"]
    print(f"✅ Carpeta encontrada: {_folders[0]['name']} (ID: {silicon_sampling_folder_id})")
else:
    raise FileNotFoundError("No se encontró la carpeta 'silicon_sampling' en Google Drive")

In [ ]:
import csv
import io

# Datos de prueba
test_data = [
    ["id", "nombre", "valor", "fecha", "categoria"],
    [1, "Alpha", 10.5, "2024-01-15", "A"],
    [2, "Beta", 20.3, "2024-01-16", "B"],
    [3, "Gamma", 15.7, "2024-01-17", "A"],
    [4, "Delta", 8.9, "2024-01-18", "C"],
    [5, "Epsilon", 33.1, "2024-01-19", "B"],
    [6, "Zeta", 12.4, "2024-01-20", "A"],
    [7, "Eta", 27.8, "2024-01-21", "C"],
    [8, "Theta", 19.2, "2024-01-22", "B"],
    [9, "Iota", 5.6, "2024-01-23", "A"],
    [10, "Kappa", 41.0, "2024-01-24", "C"],
]

# Generar el CSV en memoria
_csv_buffer = io.StringIO()
_writer = csv.writer(_csv_buffer)
_writer.writerows(test_data)
csv_content = _csv_buffer.getvalue()

print("CSV de prueba generado:")
print(csv_content)

In [ ]:
from googleapiclient.http import MediaIoBaseUpload

# Subir el CSV a la carpeta silicon_sampling
_file_metadata = {
    "name": "test_silicon_sampling.csv",
    "parents": [silicon_sampling_folder_id],
    "mimeType": "text/csv",
}

_media = MediaIoBaseUpload(
    io.BytesIO(csv_content.encode("utf-8")),
    mimetype="text/csv",
    resumable=True,
)

uploaded_file = drive_service.files().create(
    body=_file_metadata,
    media_body=_media,
    fields="id, name, size, mimeType",
).execute()

print("✅ Archivo subido a Google Drive!")
print(f"   Nombre: {uploaded_file['name']}")
print(f"   ID: {uploaded_file['id']}")
print(f"   Tamaño: {uploaded_file['size']} bytes")